# 综合题 · MLP 模块化 + 保存加载（第 5 章全章）

> 题型：**C 补全**——把第 4 章的 MLP 拆成「自定义块」，用 Sequential 组装，再走一遍保存 → 加载 → 验证。
> 覆盖第 5 章核心：nn.Module 块、参数管理（state_dict）、读写文件。
>
> 难度：🚀 挑战
> 做题流程：按第一步 → 第五步补全 TODO（共 3 处）；做完看 `solutions/综合题-答案.md`。

## 第一步 · 自定义块（TODO 1）

第 5 章的核心思想：任何网络都可以抽象成「块（block）」——一个 `nn.Module`，只要实现 `__init__`（声明层）和 `forward`（定义前向）即可。

先补全一个自定义 MLP 块：两个全连接层 + ReLU。

In [3]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)

class MLPBlock(nn.Module):
    def __init__(self, in_units, hidden_units):
        super().__init__()
        # TODO 1a: 声明两个线性层 self.hidden 和 self.out
        self.hidden = nn.Linear(in_units, hidden_units)
        self.out = nn.Linear(hidden_units, in_units)

    def forward(self, X):
        # TODO 1b: hidden → ReLU → out
        return self.out(F.relu(self.hidden(X)))

### 自测：完成 TODO 1 后运行

In [4]:
try:
    blk = MLPBlock(10, 32)
    out = blk(torch.randn(2, 10))
    assert list(out.shape) == [2, 10], f'输出形状不对: {out.shape}'
    print('✓ MLPBlock 输出形状正确:', list(out.shape))
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ MLPBlock 输出形状正确: [2, 10]


## 第二步 · 用 Sequential 组装（TODO 2）

自定义块和内置层一样，都能塞进 `nn.Sequential`。补全：一个更大的网络 = 自定义块 + ReLU + 输出层。

In [7]:
# TODO 2: 组装 net = Sequential(MLPBlock(...), nn.ReLU(), nn.Linear(...))
net = nn.Sequential(MLPBlock(10, 64), nn.ReLU(), nn.Linear(10, 10))

X = torch.randn(4, 10)
out = net(X)
print('net 输出形状:', list(out.shape))

net 输出形状: [4, 10]


## 第三步 · 读 state_dict（读代码）

`state_dict` 存的是模型所有参数的「名字 → 张量」字典，是保存/加载的核心。先预测：里面会有几个条目？键名长什么样？

In [8]:
print('state_dict 的键:')
for k, v in net.state_dict().items():
    print('  ', k, '->', list(v.shape))

state_dict 的键:
   0.hidden.weight -> [64, 10]
   0.hidden.bias -> [64]
   0.out.weight -> [10, 64]
   0.out.bias -> [10]
   2.weight -> [10, 10]
   2.bias -> [10]


## 第四步 · 保存与加载（TODO 3）

训练好的模型要能保存、能加载、加载后结果一致。补全保存 + 加载。

In [15]:
# TODO 3a: 用 torch.save 保存 net 的 state_dict 到 'net.params'
torch.save(net.state_dict(), 'net.params')

# 新建一个「结构相同但参数随机」的网络，加载刚保存的 state_dict
net2 = None
# TODO 3b: 重新组装一个与 net 结构相同的 net2，用 load_state_dict 加载保存的参数
net2 = nn.Sequential(MLPBlock(10, 64), nn.ReLU(), nn.Linear(10, 10))
net2.load_state_dict(torch.load('net.params', weights_only = False))

<All keys matched successfully>

规范性问题：
UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 

	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
    
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL getattr was not an allowed global by default. Please use `torch.serialization.add_safe_globals([getattr])` or the `torch.serialization.safe_globals([getattr])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

所以必须要多加一个weights_only = False

## 第五步 · 验证（读代码 + 运行）

加载后的 net2 应该和原来的 net 参数完全一致、前向输出完全一致。

In [16]:
# 验证参数一致
for (k1, v1), (k2, v2) in zip(net.state_dict().items(), net2.state_dict().items()):
    assert torch.equal(v1, v2), f'参数不一致: {k1}'
print('✓ 参数完全一致')

# 验证前向输出一致
net.eval()
net2.eval()
with torch.no_grad():
    out1 = net(X)
    out2 = net2(X)
assert torch.allclose(out1, out2), '前向输出不一致'
print('✓ 前向输出一致')

✓ 参数完全一致
✓ 前向输出一致


## 小结与面试衔接

- 自定义块 = 实现 `__init__`（声明层）+ `forward`（前向）的 `nn.Module` 子类
- Sequential 可以嵌套任意块（内置层、自定义块、别的 Sequential）
- state_dict 是「参数名 → 张量」字典，`torch.save(net.state_dict(), ...)` 保存、`load_state_dict` 加载
- 加载时**新网络的架构必须和保存时完全一致**，否则键对不上会报错
- 面试高频：自定义层（无参层 vs 带参层）、参数共享（两个层用同一个 nn.Linear 对象）、state_dict 里有什么